# KFP Pipeline: Fraud Detection (Feast → Validate → Train → Evaluate → Gate → Deploy)

This notebook defines and submits a **Kubeflow Pipeline** that orchestrates the full fraud-detection MLOps workflow:

| Step | Component | What it does |
|------|-----------|-------------|
| 1 | **Fetch Feast Features** | Connects to the remote Feast offline store, fetches historical features, builds train/val splits, uploads to MinIO |
| 2 | **Validate Data Quality** | Downloads the dataset, checks for nulls, class imbalance, feature statistics — blocks the pipeline if quality is too poor |
| 3 | **Train FraudMLP** | Downloads data from MinIO, trains a PyTorch MLP with dropout, uploads model + metrics to MinIO |
| 4 | **Evaluate Model** | Loads the trained model, computes accuracy / precision / recall / F1 / ROC AUC / confusion matrix, uploads a report |
| 5 | **Quality Gate** | Reads evaluation metrics and blocks deployment if the model doesn't meet the minimum AUC threshold |
| 6 | **Deploy KServe** | Creates or patches a KServe InferenceService (conditional on toggle + quality gate) |

Every training knob (epochs, learning rate, batch size, …) and every infrastructure endpoint (Feast URLs, MinIO credentials, KServe namespace, …) is exposed as a **pipeline parameter** — visible and editable in the KFP UI.

## 1) Install dependencies

In [ ]:
!pip install -q kfp

## 2) Define pipeline components

Each component is self-contained: imports and dependencies are declared inline so that KFP can serialize them into independent container steps.

In [ ]:
from kfp import dsl

### Component 1 — Fetch features from Feast

Connects to the remote Feast offline store via Arrow Flight, retrieves all fraud features for the requested date range, and produces a train/val parquet artifact.

In [ ]:
@dsl.component(
    base_image="registry.access.redhat.com/ubi9/python-311:latest",
    packages_to_install=[
        "feast[postgres,grpc]",
        "psycopg2-binary",
        "pyarrow",
        "pandas",
        "grpcio",
        "scikit-learn",
        "boto3",
    ],
)
def fetch_features(
    feast_registry_url: str,
    feast_offline_host: str,
    feast_offline_port: int,
    feast_start_date: str,
    feast_end_date: str,
    val_split: float,
    minio_endpoint: str,
    minio_access_key: str,
    minio_secret_key: str,
    minio_bucket: str,
    data_s3_key: str,
):
    """Connect to the remote Feast offline store, build train/val splits, upload to MinIO."""
    from datetime import datetime

    import boto3
    import pandas as pd
    from botocore.client import Config
    from feast import FeatureStore, RepoConfig
    from sklearn.model_selection import train_test_split

    config = RepoConfig(
        project="fraud_detection",
        provider="local",
        registry={"registry_type": "remote", "path": feast_registry_url},
        offline_store={
            "type": "remote",
            "host": feast_offline_host,
            "port": feast_offline_port,
        },
        online_store={"type": "sqlite", "path": "/tmp/feast_online.db"},
        entity_key_serialization_version=3,
    )
    store = FeatureStore(config=config)

    features = [
        "fraud_features:distance_from_home",
        "fraud_features:distance_from_last_transaction",
        "fraud_features:ratio_to_median_purchase_price",
        "fraud_features:repeat_retailer",
        "fraud_features:used_chip",
        "fraud_features:used_pin_number",
        "fraud_features:online_order",
        "fraud_features:fraud",
    ]

    start_dt = datetime.strptime(feast_start_date, "%Y-%m-%d")
    end_dt = datetime.strptime(feast_end_date, "%Y-%m-%d")

    retrieval = store.get_historical_features(
        entity_df=None,
        features=features,
        start_date=start_dt,
        end_date=end_dt,
    )
    df = retrieval.to_df()

    feature_cols = [
        "distance_from_home",
        "distance_from_last_transaction",
        "ratio_to_median_purchase_price",
        "repeat_retailer",
        "used_chip",
        "used_pin_number",
        "online_order",
    ]
    label_col = "fraud"
    df = df.dropna(subset=feature_cols + [label_col])

    train_df, val_df = train_test_split(
        df, test_size=val_split, random_state=42, stratify=df[label_col]
    )
    train_df["split"] = "train"
    val_df["split"] = "val"
    combined = pd.concat([train_df, val_df], ignore_index=True)

    local_path = "/tmp/training_data.parquet"
    combined.to_parquet(local_path, index=False)

    s3 = boto3.client(
        "s3",
        endpoint_url=minio_endpoint,
        aws_access_key_id=minio_access_key,
        aws_secret_access_key=minio_secret_key,
        config=Config(signature_version="s3v4"),
        region_name="us-east-1",
    )
    try:
        s3.create_bucket(Bucket=minio_bucket)
    except Exception:
        pass
    s3.upload_file(local_path, minio_bucket, data_s3_key)

    print(
        f"Fetched {len(df)} rows ({feast_start_date} → {feast_end_date}), "
        f"train={len(train_df)}, val={len(val_df)}"
    )
    print(f"Uploaded to s3://{minio_bucket}/{data_s3_key}")

### Component 2 — Validate Data Quality

Downloads the training data from MinIO and runs automated quality checks: row count, null percentages, class balance (fraud rate), and per-feature statistics. Uploads a JSON validation report to MinIO and **blocks the pipeline** if critical issues are found.

In [ ]:
@dsl.component(
    base_image="registry.access.redhat.com/ubi9/python-311:latest",
    packages_to_install=["pandas", "pyarrow", "boto3"],
)
def validate_data(
    data_s3_key: str,
    minio_endpoint: str,
    minio_access_key: str,
    minio_secret_key: str,
    minio_bucket: str,
    minio_model_prefix: str,
):
    """Download training data from MinIO and run quality checks."""
    import json

    import boto3
    import pandas as pd
    from botocore.client import Config

    s3 = boto3.client(
        "s3",
        endpoint_url=minio_endpoint,
        aws_access_key_id=minio_access_key,
        aws_secret_access_key=minio_secret_key,
        config=Config(signature_version="s3v4"),
        region_name="us-east-1",
    )

    local_data = "/tmp/training_data.parquet"
    s3.download_file(minio_bucket, data_s3_key, local_data)
    df = pd.read_parquet(local_data)

    feature_cols = [
        "distance_from_home",
        "distance_from_last_transaction",
        "ratio_to_median_purchase_price",
        "repeat_retailer",
        "used_chip",
        "used_pin_number",
        "online_order",
    ]
    label_col = "fraud"

    train_df = df[df["split"] == "train"]
    val_df = df[df["split"] == "val"]

    total_rows = len(df)
    null_pct = df[feature_cols + [label_col]].isnull().mean().to_dict()
    fraud_rate = float(df[label_col].mean())
    max_null = float(max(null_pct.values()))

    stats = {}
    for col in feature_cols:
        stats[col] = {
            "mean": float(df[col].mean()),
            "std": float(df[col].std()),
            "min": float(df[col].min()),
            "max": float(df[col].max()),
        }

    report = {
        "total_rows": total_rows,
        "train_rows": len(train_df),
        "val_rows": len(val_df),
        "fraud_rate": fraud_rate,
        "null_percentages": {k: float(v) for k, v in null_pct.items()},
        "feature_statistics": stats,
        "passed": True,
        "issues": [],
    }

    if total_rows < 100:
        report["issues"].append(f"Too few rows: {total_rows}")
        report["passed"] = False
    if fraud_rate < 0.01 or fraud_rate > 0.99:
        report["issues"].append(f"Severe class imbalance: fraud_rate={fraud_rate:.4f}")
    if max_null > 0.10:
        report["issues"].append(f"High null rate: {max_null:.2%}")
        report["passed"] = False

    report_path = "/tmp/validation_report.json"
    with open(report_path, "w") as f:
        json.dump(report, f, indent=2)
    s3.upload_file(
        report_path, minio_bucket, f"{minio_model_prefix}/validation_report.json"
    )

    print("=" * 60)
    print("DATA VALIDATION REPORT")
    print("=" * 60)
    print(f"  Total rows:  {total_rows:,}")
    print(f"  Train rows:  {len(train_df):,}")
    print(f"  Val rows:    {len(val_df):,}")
    print(f"  Fraud rate:  {fraud_rate:.2%}")
    print(f"  Max null %:  {max_null:.2%}")
    print("-" * 60)
    print("  Feature Statistics:")
    for col, s in stats.items():
        print(
            f"    {col:40s}  mean={s['mean']:8.2f}  "
            f"std={s['std']:8.2f}  [{s['min']:.2f}, {s['max']:.2f}]"
        )
    print("-" * 60)
    if report["issues"]:
        for issue in report["issues"]:
            print(f"  WARNING: {issue}")
    verdict = "PASSED" if report["passed"] else "FAILED"
    print(f"  VERDICT: {verdict}")
    print("=" * 60)

    if not report["passed"]:
        raise RuntimeError(f"Data validation failed: {report['issues']}")

### Component 3 — Train PyTorch FraudMLP

Reads the validated parquet dataset from MinIO, trains a `FraudMLP` (MLP with dropout), evaluates on the validation split, and uploads both the PyTorch state dict and an sklearn-compatible `model.joblib` to MinIO.

In [ ]:
@dsl.component(
    base_image="registry.access.redhat.com/ubi9/python-311:latest",
    packages_to_install=[
        "torch",
        "pandas",
        "scikit-learn",
        "pyarrow",
        "boto3",
    ],
)
def train_model(
    data_s3_key: str,
    num_epochs: int,
    batch_size: int,
    learning_rate: float,
    hidden_dim: int,
    minio_endpoint: str,
    minio_access_key: str,
    minio_secret_key: str,
    minio_bucket: str,
    minio_model_prefix: str,
):
    """Download data from MinIO, train FraudMLP, upload model to MinIO."""
    import json
    import os

    import boto3
    import numpy as np
    import pandas as pd
    import torch
    from botocore.client import Config
    from sklearn.metrics import roc_auc_score
    from torch import nn
    from torch.utils.data import DataLoader, Dataset

    np.random.seed(42)
    torch.manual_seed(42)

    s3 = boto3.client(
        "s3",
        endpoint_url=minio_endpoint,
        aws_access_key_id=minio_access_key,
        aws_secret_access_key=minio_secret_key,
        config=Config(signature_version="s3v4"),
        region_name="us-east-1",
    )

    local_data = "/tmp/training_data.parquet"
    s3.download_file(minio_bucket, data_s3_key, local_data)
    print(f"Downloaded s3://{minio_bucket}/{data_s3_key}")

    feature_cols = [
        "distance_from_home",
        "distance_from_last_transaction",
        "ratio_to_median_purchase_price",
        "repeat_retailer",
        "used_chip",
        "used_pin_number",
        "online_order",
    ]
    label_col = "fraud"

    df = pd.read_parquet(local_data)
    train_df = df[df["split"] == "train"]
    val_df = df[df["split"] == "val"]

    class TabularDataset(Dataset):
        def __init__(self, frame, fcols, lcol):
            self.x = torch.tensor(frame[fcols].values, dtype=torch.float32)
            self.y = torch.tensor(
                frame[lcol].values, dtype=torch.float32
            ).unsqueeze(1)

        def __len__(self):
            return len(self.x)

        def __getitem__(self, i):
            return self.x[i], self.y[i]

    class FraudMLP(nn.Module):
        def __init__(self, d, h):
            super().__init__()
            self.net = nn.Sequential(
                nn.Linear(d, h),
                nn.ReLU(),
                nn.Dropout(0.2),
                nn.Linear(h, h // 2),
                nn.ReLU(),
                nn.Linear(h // 2, 1),
            )

        def forward(self, x):
            return self.net(x)

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = FraudMLP(len(feature_cols), hidden_dim).to(device)

    train_loader = DataLoader(
        TabularDataset(train_df, feature_cols, label_col),
        batch_size=batch_size,
        shuffle=True,
    )
    val_loader = DataLoader(
        TabularDataset(val_df, feature_cols, label_col),
        batch_size=batch_size,
        shuffle=False,
    )

    criterion = nn.BCEWithLogitsLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)

    best_auc = 0.0
    for epoch in range(num_epochs):
        model.train()
        running_loss = 0.0
        for xb, yb in train_loader:
            xb, yb = xb.to(device), yb.to(device)
            optimizer.zero_grad()
            loss = criterion(model(xb), yb)
            loss.backward()
            optimizer.step()
            running_loss += loss.item()

        model.eval()
        targets, scores = [], []
        with torch.no_grad():
            for xb, yb in val_loader:
                xb = xb.to(device)
                scores.extend(
                    torch.sigmoid(model(xb)).cpu().numpy().reshape(-1).tolist()
                )
                targets.extend(yb.numpy().reshape(-1).tolist())

        auc = (
            roc_auc_score(targets, scores)
            if len(set(int(v) for v in targets)) >= 2
            else float("nan")
        )
        best_auc = max(best_auc, auc)
        avg_loss = running_loss / max(len(train_loader), 1)
        print(
            f"Epoch {epoch + 1}/{num_epochs} | "
            f"loss={avg_loss:.4f} | val_auc={auc:.4f}"
        )

    out_dir = "/tmp/model_output"
    os.makedirs(out_dir, exist_ok=True)
    model_path = os.path.join(out_dir, "fraud_mlp_state_dict.pt")
    torch.save(
        {
            "model_state_dict": model.state_dict(),
            "feature_columns": feature_cols,
            "label_column": label_col,
            "hidden_dim": hidden_dim,
        },
        model_path,
    )

    from sklearn.neural_network import MLPClassifier
    import joblib

    clf = MLPClassifier(
        hidden_layer_sizes=(hidden_dim, hidden_dim // 2),
        activation="relu",
        max_iter=1,
    )
    dummy_X = np.zeros((2, len(feature_cols)))
    dummy_y = np.array([0, 1])
    clf.fit(dummy_X, dummy_y)

    model.eval()
    clf.coefs_ = [
        model.net[0].weight.detach().cpu().numpy().T,
        model.net[3].weight.detach().cpu().numpy().T,
        model.net[5].weight.detach().cpu().numpy().T,
    ]
    clf.intercepts_ = [
        model.net[0].bias.detach().cpu().numpy(),
        model.net[3].bias.detach().cpu().numpy(),
        model.net[5].bias.detach().cpu().numpy(),
    ]
    joblib.dump(clf, os.path.join(out_dir, "model.joblib"))
    print(f"Exported sklearn-compatible model.joblib")

    metrics_path = os.path.join(out_dir, "metrics.json")
    with open(metrics_path, "w") as f:
        json.dump(
            {
                "val_auc": float(best_auc),
                "epochs": num_epochs,
                "train_rows": len(train_df),
                "val_rows": len(val_df),
            },
            f,
            indent=2,
        )

    for fname in os.listdir(out_dir):
        local = os.path.join(out_dir, fname)
        if os.path.isfile(local):
            key = f"{minio_model_prefix}/{fname}"
            s3.upload_file(local, minio_bucket, key)
            print(f"Uploaded s3://{minio_bucket}/{key} ({os.path.getsize(local):,} bytes)")

    model_uri = f"s3://{minio_bucket}/{minio_model_prefix}"
    print(f"Model URI: {model_uri}, best AUC={best_auc:.4f}")

### Component 4 — Evaluate Model

Loads the trained `model.joblib` and the validation split from MinIO. Computes a full classification report: accuracy, precision, recall, F1, ROC AUC, and confusion matrix. Uploads `evaluation_report.json` to MinIO.

In [ ]:
@dsl.component(
    base_image="registry.access.redhat.com/ubi9/python-311:latest",
    packages_to_install=["pandas", "pyarrow", "scikit-learn", "boto3"],
)
def evaluate_model(
    data_s3_key: str,
    minio_endpoint: str,
    minio_access_key: str,
    minio_secret_key: str,
    minio_bucket: str,
    minio_model_prefix: str,
):
    """Load trained model + validation data, compute comprehensive metrics."""
    import json

    import boto3
    import joblib
    import pandas as pd
    from botocore.client import Config
    from sklearn.metrics import (
        accuracy_score,
        classification_report,
        confusion_matrix,
        f1_score,
        precision_score,
        recall_score,
        roc_auc_score,
    )

    s3 = boto3.client(
        "s3",
        endpoint_url=minio_endpoint,
        aws_access_key_id=minio_access_key,
        aws_secret_access_key=minio_secret_key,
        config=Config(signature_version="s3v4"),
        region_name="us-east-1",
    )

    local_data = "/tmp/training_data.parquet"
    local_model = "/tmp/model.joblib"
    s3.download_file(minio_bucket, data_s3_key, local_data)
    s3.download_file(minio_bucket, f"{minio_model_prefix}/model.joblib", local_model)

    feature_cols = [
        "distance_from_home",
        "distance_from_last_transaction",
        "ratio_to_median_purchase_price",
        "repeat_retailer",
        "used_chip",
        "used_pin_number",
        "online_order",
    ]
    label_col = "fraud"

    df = pd.read_parquet(local_data)
    val_df = df[df["split"] == "val"]
    X_val = val_df[feature_cols].values
    y_val = val_df[label_col].values.astype(int)

    clf = joblib.load(local_model)
    y_pred = clf.predict(X_val)
    y_prob = clf.predict_proba(X_val)[:, 1]

    acc = accuracy_score(y_val, y_pred)
    prec = precision_score(y_val, y_pred, zero_division=0)
    rec = recall_score(y_val, y_pred, zero_division=0)
    f1 = f1_score(y_val, y_pred, zero_division=0)
    auc = roc_auc_score(y_val, y_prob) if len(set(y_val)) >= 2 else 0.0
    cm = confusion_matrix(y_val, y_pred).tolist()
    cls_report = classification_report(
        y_val, y_pred, target_names=["Legit", "Fraud"], zero_division=0
    )

    report = {
        "accuracy": float(acc),
        "precision": float(prec),
        "recall": float(rec),
        "f1_score": float(f1),
        "roc_auc": float(auc),
        "confusion_matrix": cm,
        "val_samples": len(y_val),
    }

    report_path = "/tmp/evaluation_report.json"
    with open(report_path, "w") as f:
        json.dump(report, f, indent=2)
    s3.upload_file(
        report_path, minio_bucket, f"{minio_model_prefix}/evaluation_report.json"
    )

    print("=" * 60)
    print("MODEL EVALUATION REPORT")
    print("=" * 60)
    print(f"  Validation samples:  {len(y_val):,}")
    print(f"  Accuracy:            {acc:.4f}")
    print(f"  Precision:           {prec:.4f}")
    print(f"  Recall:              {rec:.4f}")
    print(f"  F1 Score:            {f1:.4f}")
    print(f"  ROC AUC:             {auc:.4f}")
    print("-" * 60)
    print("  Confusion Matrix:")
    print(f"    TN={cm[0][0]:,}  FP={cm[0][1]:,}")
    print(f"    FN={cm[1][0]:,}  TP={cm[1][1]:,}")
    print("-" * 60)
    print(cls_report)
    print("=" * 60)

### Component 5 — Model Quality Gate

Reads the evaluation report from MinIO and enforces a minimum ROC AUC threshold. If the model falls below the threshold the step **raises an error** and the pipeline stops before deployment. The threshold is exposed as a pipeline parameter (`min_auc`).

In [ ]:
@dsl.component(
    base_image="registry.access.redhat.com/ubi9/python-311:latest",
    packages_to_install=["boto3"],
)
def check_model_quality(
    minio_endpoint: str,
    minio_access_key: str,
    minio_secret_key: str,
    minio_bucket: str,
    minio_model_prefix: str,
    min_auc: float,
):
    """Read evaluation metrics from MinIO and enforce quality threshold."""
    import json

    import boto3
    from botocore.client import Config

    s3 = boto3.client(
        "s3",
        endpoint_url=minio_endpoint,
        aws_access_key_id=minio_access_key,
        aws_secret_access_key=minio_secret_key,
        config=Config(signature_version="s3v4"),
        region_name="us-east-1",
    )

    local_report = "/tmp/evaluation_report.json"
    s3.download_file(
        minio_bucket, f"{minio_model_prefix}/evaluation_report.json", local_report
    )

    with open(local_report) as f:
        report = json.load(f)

    auc = report["roc_auc"]
    f1 = report["f1_score"]
    acc = report["accuracy"]
    prec = report["precision"]
    rec = report["recall"]

    print("=" * 60)
    print("MODEL QUALITY GATE")
    print("=" * 60)
    print(f"  ROC AUC:     {auc:.4f}  (threshold: >= {min_auc})")
    print(f"  F1 Score:    {f1:.4f}")
    print(f"  Precision:   {prec:.4f}")
    print(f"  Recall:      {rec:.4f}")
    print(f"  Accuracy:    {acc:.4f}")
    print("-" * 60)

    if auc >= min_auc:
        print(f"  PASSED — AUC {auc:.4f} meets threshold {min_auc}")
        print("  Model approved for deployment.")
    else:
        print(f"  FAILED — AUC {auc:.4f} below threshold {min_auc}")
        print("  Model rejected. Deployment blocked.")
        print("=" * 60)
        raise RuntimeError(
            f"Quality gate failed: AUC={auc:.4f} < min_auc={min_auc}"
        )
    print("=" * 60)

### Component 6 — Deploy / update KServe InferenceService

Uses the Kubernetes Python client to create (or patch if it already exists) a KServe InferenceService that pulls the model from MinIO. Only runs if the quality gate passed and `deploy_model` is `True`.

> **Note:** Each step communicates via MinIO S3 paths (plain strings) to avoid a KFP v2.4 driver bug with artifact passing.

In [ ]:
@dsl.component(
    base_image="registry.access.redhat.com/ubi9/python-311:latest",
    packages_to_install=["kubernetes"],
)
def deploy_kserve(
    minio_bucket: str,
    minio_model_prefix: str,
    kserve_namespace: str,
    inference_service_name: str,
    model_format: str,
):
    """Create or patch a KServe InferenceService pointing at the MinIO model."""
    from kubernetes import client, config

    model_uri = f"s3://{minio_bucket}/{minio_model_prefix}"
    print(f"Deploying InferenceService '{inference_service_name}' in {kserve_namespace}")
    print(f"Storage URI: {model_uri}")

    try:
        config.load_incluster_config()
    except config.ConfigException:
        config.load_kube_config()

    api = client.CustomObjectsApi()
    isvc = {
        "apiVersion": "serving.kserve.io/v1beta1",
        "kind": "InferenceService",
        "metadata": {
            "name": inference_service_name,
            "namespace": kserve_namespace,
        },
        "spec": {
            "predictor": {
                "serviceAccountName": "kserve-minio-sa",
                "model": {
                    "modelFormat": {"name": model_format},
                    "storageUri": model_uri,
                    "resources": {
                        "requests": {"memory": "512Mi", "cpu": "250m"},
                        "limits": {"memory": "1Gi", "cpu": "500m"},
                    },
                },
            }
        },
    }

    try:
        api.get_namespaced_custom_object(
            group="serving.kserve.io",
            version="v1beta1",
            namespace=kserve_namespace,
            plural="inferenceservices",
            name=inference_service_name,
        )
        api.patch_namespaced_custom_object(
            group="serving.kserve.io",
            version="v1beta1",
            namespace=kserve_namespace,
            plural="inferenceservices",
            name=inference_service_name,
            body=isvc,
        )
        print(f"Patched existing InferenceService '{inference_service_name}'")
    except client.exceptions.ApiException as e:
        if e.status == 404:
            api.create_namespaced_custom_object(
                group="serving.kserve.io",
                version="v1beta1",
                namespace=kserve_namespace,
                plural="inferenceservices",
                body=isvc,
            )
            print(f"Created InferenceService '{inference_service_name}'")
        else:
            raise

## 3) Define the pipeline

All parameters appear in the KFP "Create Run" UI. Users can override any default before submitting.

**Parameter groups:**
- **Feast** — registry URL, offline store host/port, date range for feature retrieval
- **Training** — epochs, batch size, learning rate, hidden dim, validation split
- **MinIO** — endpoint, credentials, bucket, model prefix, data key
- **Quality gate** — minimum AUC threshold to allow deployment
- **KServe** — deploy toggle, namespace, service name, model format

In [ ]:
@dsl.pipeline(
    name="Fraud Detection — Feast → Validate → Train → Evaluate → Gate → Deploy",
    description=(
        "End-to-end fraud detection MLOps pipeline. Fetches features from a "
        "remote Feast offline store, validates data quality, trains a PyTorch "
        "MLP, evaluates the model with full metrics, enforces a quality gate "
        "(minimum AUC), and optionally deploys a KServe InferenceService. "
        "All knobs are exposed as pipeline parameters."
    ),
)
def fraud_detection_pipeline(
    # ── Feast parameters ──────────────────────────────────────────────
    feast_registry_url: str = "feast-registry-service.mlops-workshop.svc.cluster.local:6567",
    feast_offline_host: str = "feast-offline-service.mlops-workshop.svc.cluster.local",
    feast_offline_port: int = 8815,
    feast_start_date: str = "2025-01-01",
    feast_end_date: str = "2025-03-31",
    # ── Training hyper-parameters ─────────────────────────────────────
    num_epochs: int = 5,
    batch_size: int = 256,
    learning_rate: float = 0.001,
    hidden_dim: int = 64,
    val_split: float = 0.2,
    # ── MinIO / S3 parameters ─────────────────────────────────────────
    minio_endpoint: str = "http://minio-service.kubeflow.svc.cluster.local:9000",
    minio_access_key: str = "minio",
    minio_secret_key: str = "minio123",
    minio_bucket: str = "models",
    minio_model_prefix: str = "fraud-detector",
    data_s3_key: str = "pipeline-data/training_data.parquet",
    # ── Quality gate ──────────────────────────────────────────────────
    min_auc: float = 0.7,
    # ── KServe parameters ─────────────────────────────────────────────
    deploy_model: bool = True,
    kserve_namespace: str = "mlops-workshop",
    inference_service_name: str = "fraud-detector",
    model_format: str = "sklearn",
):
    # Step 1 — Fetch features from Feast and upload to MinIO
    fetch_task = fetch_features(
        feast_registry_url=feast_registry_url,
        feast_offline_host=feast_offline_host,
        feast_offline_port=feast_offline_port,
        feast_start_date=feast_start_date,
        feast_end_date=feast_end_date,
        val_split=val_split,
        minio_endpoint=minio_endpoint,
        minio_access_key=minio_access_key,
        minio_secret_key=minio_secret_key,
        minio_bucket=minio_bucket,
        data_s3_key=data_s3_key,
    ).set_display_name("1. Fetch Feast Features")

    # Step 2 — Validate data quality
    validate_task = validate_data(
        data_s3_key=data_s3_key,
        minio_endpoint=minio_endpoint,
        minio_access_key=minio_access_key,
        minio_secret_key=minio_secret_key,
        minio_bucket=minio_bucket,
        minio_model_prefix=minio_model_prefix,
    ).after(fetch_task).set_display_name("2. Validate Data Quality")

    # Step 3 — Train PyTorch FraudMLP
    train_task = train_model(
        data_s3_key=data_s3_key,
        num_epochs=num_epochs,
        batch_size=batch_size,
        learning_rate=learning_rate,
        hidden_dim=hidden_dim,
        minio_endpoint=minio_endpoint,
        minio_access_key=minio_access_key,
        minio_secret_key=minio_secret_key,
        minio_bucket=minio_bucket,
        minio_model_prefix=minio_model_prefix,
    ).after(validate_task).set_display_name("3. Train FraudMLP")

    # Step 4 — Evaluate model with full metrics
    eval_task = evaluate_model(
        data_s3_key=data_s3_key,
        minio_endpoint=minio_endpoint,
        minio_access_key=minio_access_key,
        minio_secret_key=minio_secret_key,
        minio_bucket=minio_bucket,
        minio_model_prefix=minio_model_prefix,
    ).after(train_task).set_display_name("4. Evaluate Model")

    # Step 5 — Enforce quality gate
    gate_task = check_model_quality(
        minio_endpoint=minio_endpoint,
        minio_access_key=minio_access_key,
        minio_secret_key=minio_secret_key,
        minio_bucket=minio_bucket,
        minio_model_prefix=minio_model_prefix,
        min_auc=min_auc,
    ).after(eval_task).set_display_name("5. Quality Gate")

    # Step 6 — Deploy to KServe (conditional)
    with dsl.If(deploy_model == True, name="Deploy if Approved"):  # noqa: E712
        deploy_kserve(
            minio_bucket=minio_bucket,
            minio_model_prefix=minio_model_prefix,
            kserve_namespace=kserve_namespace,
            inference_service_name=inference_service_name,
            model_format=model_format,
        ).after(gate_task).set_display_name("6. Deploy KServe InferenceService")

print("Pipeline defined.")

## 4) Compile pipeline to YAML

This generates a portable `fraud_detection_pipeline.yaml` that can be uploaded to any KFP instance.

In [ ]:
from kfp import compiler

YAML_PATH = "fraud_detection_pipeline.yaml"

compiler.Compiler().compile(
    pipeline_func=fraud_detection_pipeline,
    package_path=YAML_PATH,
)
print(f"Compiled → {YAML_PATH}")

## 5) Submit pipeline run

Connect to the KFP API server and submit a run. Override any defaults below before executing.

In [ ]:
import kfp

KFP_ENDPOINT = "http://kfp-ui-kubeflow.apps.rosa.k2s7v9j3f3g5j9l.yqif.p3.openshiftapps.com"
EXPERIMENT_NAME = "fraud-detection-feast"
RUN_NAME = "fraud-feast-run-03"

kfp_client = kfp.Client(host=KFP_ENDPOINT)
print(f"Connected to KFP at {KFP_ENDPOINT}")
print(f"KFP SDK version: {kfp.__version__}")

### Override parameters for this run

Edit any values below before submitting. These are the same parameters that appear in the KFP UI.

In [ ]:
PIPELINE_PARAMS = {
    # ── Feast ─────────────────────────────────────────────────────────
    "feast_registry_url": "feast-registry-service.mlops-workshop.svc.cluster.local:6567",
    "feast_offline_host": "feast-offline-service.mlops-workshop.svc.cluster.local",
    "feast_offline_port": 8815,
    "feast_start_date": "2025-01-01",
    "feast_end_date": "2025-03-31",
    # ── Training ──────────────────────────────────────────────────────
    "num_epochs": 5,
    "batch_size": 256,
    "learning_rate": 0.001,
    "hidden_dim": 64,
    "val_split": 0.2,
    # ── MinIO ─────────────────────────────────────────────────────────
    "minio_endpoint": "http://minio-service.kubeflow.svc.cluster.local:9000",
    "minio_access_key": "minio",
    "minio_secret_key": "minio123",
    "minio_bucket": "models",
    "minio_model_prefix": "fraud-detector",
    "data_s3_key": "pipeline-data/training_data.parquet",
    # ── Quality gate ──────────────────────────────────────────────────
    "min_auc": 0.7,
    # ── KServe ────────────────────────────────────────────────────────
    "deploy_model": True,
    "kserve_namespace": "mlops-workshop",
    "inference_service_name": "fraud-detector",
    "model_format": "sklearn",
}

print("Pipeline parameters:")
for k, v in PIPELINE_PARAMS.items():
    print(f"  {k:30s} = {v}")

In [ ]:
run = kfp_client.create_run_from_pipeline_package(
    pipeline_file=YAML_PATH,
    arguments=PIPELINE_PARAMS,
    run_name=RUN_NAME,
    experiment_name=EXPERIMENT_NAME,
)

print(f"Run submitted: {run.run_id}")
print(f"Experiment:    {EXPERIMENT_NAME}")
print(f"View in UI:    {KFP_ENDPOINT}/#/runs/details/{run.run_id}")

## 6) Monitor run

In [ ]:
kfp_client.wait_for_run_completion(run_id=run.run_id, timeout=900)
print("Pipeline run completed.")